# Fireworks LLM Evaluation
This notebook loads `.env`, reads tasks/truth/results, and uses a Fireworks model as an LLM judge.

In [1]:
%pip install -q openai python-dotenv pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import os, json
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key=os.getenv("FIREWORKS_API_KEY")
base_url="https://api.fireworks.ai/inference/v1"
model=os.getenv("EVAL_MODEL") or os.getenv("MODEL") or os.getenv("ALLOWED_MODELS","accounts/fireworks/models/llama-v3p1-8b-instruct")

if "," in model:
    model=model.split(",")[0].strip()

client=OpenAI(api_key=api_key,base_url=base_url)

ROOT=Path("..")
OUT=ROOT/"output"
IN=ROOT/"input"
with open(IN/"tasks.json","r",encoding="utf-8") as f:
    tasks={x["task_id"]:x["prompt"] for x in json.load(f)}
with open(OUT/"truth.json","r",encoding="utf-8") as f:
    truth={x["task_id"]:x["answer"] for x in json.load(f)}
with open(OUT/"results.json","r",encoding="utf-8") as f:
    results={x["task_id"]:x["answer"] for x in json.load(f)}

print("Using model:",model)
print("Loaded",len(tasks),"tasks")


Using model: accounts/fireworks/models/minimax-m3
Loaded 8 tasks


In [9]:
import json
import re

def judge(prompt, gt, pred):
    eval_prompt = f"""
You are an impartial evaluator.

Task:
{prompt}

Reference Answer:
{gt}

Student Answer:
{pred}

Score the student's answer from 0 to 100.

Return ONLY valid JSON.

{{
    "score": 95,
    "reason": "Brief explanation."
}}
"""

    r = client.chat.completions.create(
        model=model,
        temperature=0,
        max_tokens=512,
        messages=[
            {
                "role": "user",
                "content": eval_prompt
            }
        ]
    )

    # Print the raw response for debugging
    print("=" * 80)
    print(r.model_dump_json(indent=2))
    print("=" * 80)

    msg = r.choices[0].message

    # Use content first, fall back to reasoning_content
    content = msg.content

    if content is None:
        content = getattr(msg, "reasoning_content", None)

    if content is None:
        raise RuntimeError(
            "The model returned neither content nor reasoning_content."
        )

    # Extract the first JSON object from the response
    match = re.search(r"\{.*\}", content, re.DOTALL)

    if not match:
        raise ValueError(f"No JSON found in:\n{content}")

    return json.loads(match.group(0))

In [10]:
print("Base URL:", base_url)
print("Model:", model)

rows = []

for tid, prompt in tasks.items():
    print(f"Evaluating {tid}...")

    ev = judge(prompt, truth[tid], results.get(tid, ""))

    rows.append({
        "task_id": tid,
        "score": ev["score"],
        "reason": ev["reason"]
    })

df = pd.DataFrame(rows)
display(df)

print("Average Score:", df["score"].mean())

df.to_csv("evaluation.csv", index=False)
print("Saved evaluation.csv")


Base URL: https://api.fireworks.ai/inference/v1
Model: accounts/fireworks/models/minimax-m3
Evaluating practice-01...
{
  "id": "chatcmpl-77020c1c02c143889d42462c4a571d18",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "{\n    \"score\": 98,\n    \"reason\": \"The student correctly identified Canberra as the capital of Australia and Lake Burley Griffin as the nearby body of water, matching the reference answer. The additional detail about it being an artificial lake in the center of the city is accurate and enhances the response.\"\n}",
        "refusal": null,
        "role": "assistant",
        "annotations": null,
        "audio": null,
        "function_call": null,
        "tool_calls": null,
        "reasoning_content": "The student correctly identified Canberra as the capital of Australia and mentioned Lake Burley Griffin as the body of water it is near. The student even added that it's an art

,task_id,score,reason
0,practice-01,98,The student correctly identified Canberra as t...
1,practice-02,100,The student's answer of 144 is correct. Monday...
2,practice-03,100,The student's answer 'Mixed' exactly matches t...
3,practice-04,92,The student's answer is a single sentence that...
4,practice-05,100,The student's answer perfectly matches the ref...
5,practice-06,100,The student correctly identified the bug (retu...
6,practice-07,100,The student's answer 'Sam' exactly matches the...
7,practice-08,95,The student's solution is functionally correct...


Average Score: 98.125
Saved evaluation.csv
